# RFQ factor variants: the production optimizer as the judge

**Why a separate notebook.** The lab builds the factor model and tests its pieces on their own terms (fit, calibration, information coefficients).
None of that decides whether a piece helps the desk quote. The only test that does is the production optimizer replayed on every request with the
piece in it, paired against production on the same requests, with the win decided by the logged cover or winning print. This notebook does only that.
Every variant is a change to one term of production's objective, `argmax over x of (x + c) p(x)`, quoted by production's own argmax on
production's own fill curves, and nothing is gated by the model-level verdicts: the replay is the source of truth on relative performance.

**Inputs.** The handoff of lab Section 15a: the request frame with the logged quote, charges, outcome, thresholds and exits; the production fill
curves; the betas, exposures, idiosyncratic variances and the picked covariance on the requests (14d); every exit estimator's markout (14f);
the model signals at the request (14 [5]); the realised holding time by bucket (17).

**The four pairs.** Each variant is quoted twice, with the manual charge in `c` and with it removed, and so is production:

| Pair | Variant quoted with | Production quoted with | Question |
|---|---|---|---|
| A | manual kept | manual kept | does the factor term add to what the traders charge |
| B | manual removed | manual kept | can the factor term replace the manual charge |
| C | manual removed | manual removed | what the factor term is worth on its own |
| D | production, manual removed | production, manual kept | the reference: what the manual charge itself is worth |

Each pair is read raw (the frontier point) and levelled (a constant width per month so the variant's win rate equals its comparator's, so cover
and wins kept are compared at equal wins). Every cell carries the same columns, paired on identical requests.

**The families.** (5) The fill curve reshaped by volatility, the consolidation of the old risk tilt and pfill refit: a horizontal stretch, a logit
tilt, and the desk's own curve by volatility tercile, each with four volatility definitions. (6) Charges in the Avellaneda-Stoikov form: the vol
half-spread, the own-CUSIP inventory term, the book inventory term (full and hedged) and the combination with the curve. (7) The model signals as
charges: state space, residual level, peer gap, filtered convergence, copula conditional PIT. (4) The references and (8) the matrix.

| Cell | Content |
|---|---|
| [1] | setup: config, cache keyed on the fields each stage reads, registry, pre-registered bars |
| [2] | the handoff, the arrays, the replay check (production reproduced bit for bit) |
| [3] | machinery: the optimizer (verbatim), requote by charge and by curve, the win rule, levelling, the sequential book, the paired scorecard |
| [4] | references: production with and without the manual charge, the width frontiers, pair D |
| [5] | fill-curve family: validation (held-out log-likelihood, calibration by vol decile), then quoted |
| [6] | charge family: half-spread, own inventory, book inventory full and hedged, combined; the walk-forward k pick |
| [7] | signal family: the five model signals as charges |
| [8] | the matrix: pairs A, B, C raw and levelled; leaderboards; bars; verdicts |
| [9] | registry and runtimes |

In [ ]:
from __future__ import annotations

import hashlib
import inspect
import json
import os
import pickle
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore', category=RuntimeWarning)
try:
    from IPython.display import display  # noqa: F401
    get_ipython()  # type: ignore[name-defined]  # noqa: F821
    _INTERACTIVE = True
except Exception:  # noqa: BLE001
    _INTERACTIVE = False

    def display(obj):  # noqa: D103
        with pd.option_context('display.width', 250, 'display.max_columns', 60, 'display.max_rows', 120):
            print(obj)
import matplotlib
if not _INTERACTIVE:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt  # noqa: E402

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)


@dataclass(frozen=True)
class VariantConfig:
    spec_version: str = 'rfq_variants_v1'
    seed: int = 7
    k_grid: tuple[int, ...] = (1, 2, 3, 5)              # median extra charge in bp for the charge family
    stretch_exponents: tuple[float, ...] = (0.5, 1.0)     # horizontal stretch of the fill curve by (sigma_ref / sigma)^a
    logit_kappas: tuple[float, ...] = (0.5, 1.0)          # logit tilt of the fill curve by kappa x z(sigma)
    n0_grid: tuple[float, ...] = (0.0, 1e1, 1e2, 1e3, 1e4, 1e5, float('inf'))   # shrinkage pseudo-counts of the tercile refit
    signal_k: tuple[float, ...] = (1.0, 2.0, 3.0)         # bp per sd of a signal, as a charge
    hedge_cost_bp: float = 1.0
    bar_t: float = 3.0
    bar_months: int = 4
    bar_wins_kept: float = 0.8
    bar_cover_saved_bp: float = 1.0
    level_range_bp: float = 40.0
    level_iters: int = 16
    width_grid: tuple[int, ...] = tuple(range(-5, 6))
    chunk: int = 20_000
    test_months: tuple[str, ...] = ()                     # empty = the lab's test months from the handoff


CFG = VariantConfig()
ROOT = Path(os.environ.get('MUNI_LAB_ROOT', '/home/k178619/muni/Algos/factor_model'))
ARTIFACTS = ROOT / 'artifacts_ipca_lab'; HANDOFF = ARTIFACTS / 'handoff_rfq_variants'
OUT = ARTIFACTS / 'variants'; CACHE_DIR = OUT / 'cache'; FIGURES = OUT / 'figures'
for _p in (CACHE_DIR, FIGURES):
    _p.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(CFG.seed)
REGISTRY: dict = {'spec_version': CFG.spec_version, 'config': asdict(CFG), 'run_at': pd.Timestamp.now('UTC').isoformat()}
STAGE_KEYS: dict[str, str] = {}; CACHE_LOG: list[dict] = []; CELL_LOG: list[dict] = []
_RUN_T0 = time.perf_counter(); _CELL_STATE = {'t0': _RUN_T0, 'label': '1. Setup [1]'}

PRE_REGISTERED: dict[str, str] = {
    'variants_pairs': ('fixed 2026-10-27 before any variant was quoted: four pairs, each paired on identical requests of the test months (the lab\'s test months: the months after the '
                       'first month with a covariance), the win decided by the logged cover or winning print as in lab Section 14 [3]: A = variant with the manual charge in c '
                       'against production; B = variant with the manual charge removed from c against production (manual kept); C = both without the manual charge; D = production '
                       'without the manual charge against production, the reference. Each pair raw and levelled, levelled = a constant width per month, set by bisection, so the '
                       'variant\'s win rate on the month\'s paired requests equals its comparator\'s. Columns in every cell: win rate, wins kept, new wins, wins given up, bp left on the '
                       'table on wins with a cover, bp saved on kept cover wins, crossing on new wins, the delta per request under the exit of record (14f), the bucket guard, the '
                       'day\'s common markout for new wins, the hedged record exit (MMD at tenor over the holding and the hedge cost), and the lab\'s other-side exit as a reference; '
                       'weekly Newey-West t and months positive under each; dollars per month; the position against the width frontier of the comparator\'s regime at the variant\'s '
                       'win rate; the ex-ante daily sd of the policy\'s own book and gross DV01'),
    'variants_bars': ('fixed 2026-10-27: pair A raw and pair C raw pass on the lab Section 14 bar under the exit of record, the guard and, for a variant that raises the win rate, the '
                      'common-markout valuation together: weekly NW t >= 3, months positive >= min(4, N), at or above the width frontier; pairs B and C levelled pass if wins kept '
                      '>= 80 percent of the comparator\'s wins, cover saved on kept cover wins >= 1 bp, and the hedged delta under the guard >= 0, in >= min(4, N) months; pair B raw '
                      'passes on the pair A rule against production\'s frontier. Every row also carries one of four labels: accepted under the specified gate; positive evidence, '
                      'gate not met (record delta t >= 2 and no sign flip under the guard or the common markout); negative evidence (t <= -2 under the record, or a sign flip under '
                      'the guard or the common markout with |t| >= 2); inconclusive. Pair D has no bar. No bar is moved after the run'),
    'variants_families': ('fixed 2026-10-27: (5) fill curve by volatility: horizontal stretch p(x (sigma_ref / sigma_i)^a), a in {0.5, 1}; logit tilt logit p + kappa z_i, kappa in '
                          '{0.5, 1}; the desk\'s own curve by volatility tercile shrunk to production with n0 by the previous month\'s log-likelihood; sigma in {systematic, '
                          'idiosyncratic, total (14d, at the horizon of record), conditional scale (lab a_risk)}; sigma_ref and z walk-forward per key from earlier months; '
                          'validated before quoting by the held-out log-likelihood of the logged outcome at the logged concession against production\'s curve and by calibration '
                          'by sigma decile, reported, not gating. (6) charges: half-spread gamma sigma_h^2 u / 2; own-CUSIP inventory gamma sigma_h^2 (Q_i + u_i / 2) with Q_i the '
                          'policy\'s own DV01 in the CUSIP over the last h days; book inventory gamma s_i (lab 14e, full) and hedged (tenor block zeroed, hedge cost added); '
                          'combined = the curve form picked walk-forward by log-likelihood plus the book charge; gamma_k = k / median s on production\'s book in the first test month, '
                          'k in {1, 2, 3, 5}; the horizon h of a request = the realised par-weighted median holding of its tenor x call bucket (17), else the lab horizon; the '
                          'walk-forward pick of k per family by the hedged weekly dollar Sharpe on earlier test months. (7) signals as charges c + k z_i, z_i the normal score of the '
                          'signal on earlier months (positive = expected to richen, the lab orientation), k in {1, 2, 3} bp per sd, 0 where the signal is missing'),
}


def CELL_T(label: str) -> None:
    now = time.perf_counter(); CELL_LOG.append({'cell': _CELL_STATE['label'], 'seconds': now - _CELL_STATE['t0']}); _CELL_STATE.update(t0=now, label=label)
    print(f'\n=== {label} | elapsed {(now - _RUN_T0) / 60:.1f} min ===')


def record(section: str, **values) -> None:
    REGISTRY.setdefault(section, {}).update({k: (v.item() if hasattr(v, 'item') else v) for k, v in values.items()})
    json.dumps(REGISTRY[section], default=str)


def savefig(name: str) -> None:
    plt.tight_layout(); plt.savefig(FIGURES / f'{name}.png', dpi=110)
    if _INTERACTIVE:
        plt.show()
    plt.close('all')


def flat_records(frame: pd.DataFrame, digits: int = 4) -> list[dict]:
    f = frame.copy()
    if isinstance(f.columns, pd.MultiIndex):
        f.columns = [' | '.join(str(x) for x in c if str(x) != '') for c in f.columns]
    return f.round(digits).reset_index().to_dict(orient='records')


def nw_t(x, lags: int | None = None) -> float:
    """Newey-West (Bartlett kernel) t statistic of the mean; lags default floor(4 (n / 100)^(2/9)). Verbatim from the lab."""
    x = np.asarray(x, float); x = x[np.isfinite(x)]; n = len(x)
    if n < 4:
        return np.nan
    L = int(np.floor(4.0 * (n / 100.0) ** (2.0 / 9.0))) if lags is None else int(lags)
    u = x - x.mean(); v = float(u @ u) / n
    for l in range(1, min(L, n - 1) + 1):
        v += 2.0 * (1.0 - l / (L + 1.0)) * float(u[l:] @ u[:-l]) / n
    return float(x.mean() / np.sqrt(v / n)) if v > 0 else np.nan


def _jsonable(x):
    if isinstance(x, dict):
        return {str(k): _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonable(v) for v in x]
    if isinstance(x, np.ndarray):
        return {'shape': list(x.shape), 'sum': float(np.nansum(x.astype(float))) if x.dtype.kind in 'fiub' else str(x[:5])}
    if hasattr(x, 'item'):
        return x.item()
    return x


def _stage_source(obj) -> str:
    try:
        return inspect.getsource(obj)
    except (OSError, TypeError):
        return repr(obj)


def _fingerprint(payload) -> str:
    return hashlib.sha1(json.dumps(_jsonable(payload), sort_keys=True, default=str).encode()).hexdigest()[:16]


def frame_fingerprint(frame: pd.DataFrame, cols: list[str] | None = None) -> str:
    f = frame if cols is None else frame[cols]
    h = pd.util.hash_pandas_object(f, index=True).to_numpy(np.uint64)
    return _fingerprint({'shape': f.shape, 'columns': list(f.columns), 'xor': int(np.bitwise_xor.reduce(h)) if len(h) else 0, 'sum': int(h.sum())})


def _stage_files(name: str) -> list[Path]:
    import re
    pat = re.compile(rf'^{re.escape(name)}_[0-9a-f]{{16}}\.(pkl|parquet)$')
    return [p for p in CACHE_DIR.glob(f'{name}_*') if pat.match(p.name)]


def cached(name: str, fn, deps=(), code=(), cfg: tuple[str, ...] = (), force: bool = False):
    """Run or load a stage. The key binds the spec, ONLY the config fields the stage reads (cfg), its data dependencies and its source, so a change
    to a field another stage reads does not rebuild this one. DataFrames go to parquet, else pickle; temporary file then rename; unreadable cache rebuilt."""
    sub = {k: getattr(CFG, k) for k in cfg}
    key = _fingerprint({'spec': CFG.spec_version, 'cfg': sub, 'deps': deps, 'code': [_stage_source(fn)] + [_stage_source(x) for x in code]})
    STAGE_KEYS[name] = key
    path_pq, path_pk = CACHE_DIR / f'{name}_{key}.parquet', CACHE_DIR / f'{name}_{key}.pkl'
    path = path_pq if path_pq.exists() else path_pk; t0 = time.perf_counter(); status = 'computed'; out = None
    if path.exists() and not force:
        try:
            out = pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_pickle(path); status = 'reused'
        except (EOFError, pickle.UnpicklingError, OSError, ValueError, AttributeError, ModuleNotFoundError) as exc:
            print(f'{name}: cache file {path.name} unreadable ({type(exc).__name__}); rebuilding'); path.unlink(missing_ok=True)
    if status != 'reused':
        out = fn()
        if isinstance(out, pd.DataFrame):
            tmp = path_pq.with_name(f'{path_pq.name}.tmp{os.getpid()}')
            try:
                out.to_parquet(tmp); os.replace(tmp, path_pq); path = path_pq
            except Exception as exc:  # noqa: BLE001
                tmp.unlink(missing_ok=True); print(f'  parquet failed ({type(exc).__name__}); stored as pickle'); tmp = path_pk.with_name(f'{path_pk.name}.tmp{os.getpid()}'); pd.to_pickle(out, tmp); os.replace(tmp, path_pk); path = path_pk
        else:
            tmp = path_pk.with_name(f'{path_pk.name}.tmp{os.getpid()}'); pd.to_pickle(out, tmp); os.replace(tmp, path_pk); path = path_pk
        for old in _stage_files(name):
            if old != path:
                old.unlink(missing_ok=True)
    dt = time.perf_counter() - t0
    CACHE_LOG.append({'stage': name, 'key': key, 'status': status, 'seconds': dt, 'file': path.name, 'cell': _CELL_STATE['label']})
    print(f'{name}: {"cache" if status == "reused" else status} {dt:.1f}s [{key}] ({path.stat().st_size / 1e6:,.0f} MB)')
    return out


for _tmp in CACHE_DIR.glob('*.tmp*'):
    _tmp.unlink(missing_ok=True)
print(f'variants {CFG.spec_version} | root {ROOT} | handoff {HANDOFF} | k grid {CFG.k_grid} | hedge cost {CFG.hedge_cost_bp} bp')

## 2. The handoff and the replay check

The request frame, the curves and the arrays as the lab left them. Production is re-quoted here by the optimizer copied verbatim from the lab and
must reproduce the logged optimal quantile and every logged win and loss; nothing is scored if it does not.

In [ ]:
CELL_T('2. The handoff and the replay check [2]')
_h_t0 = time.perf_counter()
assert HANDOFF.exists(), f'{HANDOFF} not found: run lab Section 15a first'
BT = pd.read_parquet(HANDOFF / 'requests.parquet'); _A = dict(np.load(HANDOFF / 'arrays.npz')); _O = pd.read_pickle(HANDOFF / 'objects.pkl')
META, BT_CURVES, SIGMA = _O['meta'], _O['curves'], {int(k): v for k, v in _O['sigma'].items()}
HOLD_TAB = pd.read_parquet(HANDOFF / 'holding.parquet') if (HANDOFF / 'holding.parquet').exists() else None
N = len(BT); BT['date'] = pd.to_datetime(BT['date']).dt.tz_localize(None) if getattr(BT['date'].dt, 'tz', None) is not None else pd.to_datetime(BT['date'])
BT_EPS_BP, BT_H_RECORD, BT_HOLD = float(META['eps_bp']), int(META['h_record']), int(META['hold'])
BT_MONTHS = sorted(BT['month'].unique()); TEST_M = list(CFG.test_months) or list(META.get('test_months') or [])
if not TEST_M:
    TEST_M = BT_MONTHS[1:]
_month = BT['month'].to_numpy(); _week = BT['week'].to_numpy(); _winable = BT['winnable'].to_numpy(bool)
_thr = BT['threshold_y'].to_numpy(float); _grp = BT['group'].astype(str).to_numpy()
_m_sig, _adj, _ch = BT['signalyield'].to_numpy(float), BT['adj_yield'].to_numpy(float), BT['charges_level'].to_numpy(float)
_man, _liq, _rsk = (BT[c].to_numpy(float) for c in ('manualcharges', 'liquiditycharge', 'riskcharge'))
_c_base = -(_liq + _rsk)                                   # production's charge with the manual charge removed
_y_tr = BT['y_trade'].to_numpy(float); _usd = BT['usd_per_bp'].to_numpy(float)
_keys = BT['pfill_key'].astype(str).to_numpy(); _files = BT['pfill_file'].astype(str).to_numpy()
_xlog = BT['optimalQuantileBp'].to_numpy(float); _wlog = BT['won_logged'].to_numpy(bool); _y_van = BT['y_vanilla'].to_numpy(float)
_pb_days = pd.DatetimeIndex(pd.to_datetime(META['pb_days'])); _pb_di = _pb_days.get_indexer(BT['date'].dt.normalize()); _pb_nd = len(_pb_days)
assert (_pb_di >= 0).all(), 'request dates outside the handoff day index'
_pb_rows_day = pd.Series(np.arange(N)).groupby(_pb_di).indices
_pb_u = _A['pb_u'] if 'pb_u' in _A else np.where(_winable, np.nan_to_num(_usd), 0.0)
_pb_month = np.array(_pb_days.strftime('%Y-%m'))
_tm = np.isin(_month, TEST_M); _tmw = _tm & _winable; _tdays = np.isin(_pb_month, TEST_M)
_nT, _D = int(META['nT']), int(META['D'])
EXPO, V_IDIO = _A['BT_EXPO_GZ'], _A['BT_V_C']; EXPO_H = EXPO.copy(); EXPO_H[:, :_nT] = 0.0
VOL = {k[4:]: v for k, v in _A.items() if k.startswith('VOL_')}; MEXIT = {k[6:]: v for k, v in _A.items() if k.startswith('MEXIT_')}
EX_RECORD, EX_GUARD = META['exit_record'], META['exit_guard']
assert EX_RECORD in MEXIT and EX_GUARD in MEXIT, f'exits {EX_RECORD} / {EX_GUARD} missing from the handoff'
_M_REC, _M_GUARD, _M_OTHER = MEXIT[EX_RECORD], MEXIT[EX_GUARD], BT[f'M_{BT_H_RECORD}_X'].to_numpy(float)
_mmd5 = _A['mmd5'] if 'mmd5' in _A else np.full(N, np.nan)
_cm_rec = pd.Series(np.where(_winable, _M_REC, np.nan)).groupby(BT['date'].to_numpy()).transform('median').to_numpy(float)   # the day's common markout of the record exit
SIG_NAMES = dict(META['signals']); SIG = {k: BT[f'a_{k}'].to_numpy(float) for k in SIG_NAMES if f'a_{k}' in BT}
_a_risk = BT['a_risk'].to_numpy(float) if 'a_risk' in BT else np.full(N, np.nan)
_sig_days = sorted(SIGMA); _ci = pd.factorize(BT['cusip'].astype(str))[0]; _n_cusip = int(_ci.max()) + 1
_wk_code, _wk_levels = pd.factorize(pd.Series(_week), sort=True); _mo_code, _mo_levels = pd.factorize(pd.Series(_month), sort=True)


def _group_mean(d: np.ndarray, codes: np.ndarray, n_levels: int) -> np.ndarray:
    """Mean of the finite values of d per code (NaN where a code has none); the weekly and monthly aggregation without pandas."""
    ok = np.isfinite(d); num = np.bincount(codes[ok], weights=d[ok], minlength=n_levels); den = np.bincount(codes[ok], minlength=n_levels)
    return np.where(den > 0, num / np.where(den > 0, den, 1), np.nan)


# ---- the optimizer, verbatim from lab Section 14 [1]; the win rule from 14 [3]
def bt_optimize(qbp: np.ndarray, p: np.ndarray, charges_level: np.ndarray, width: float = 0.0) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Production argmax of (quantile_bp + width + charges) x p for each request, solved once per distinct charge level."""
    c = np.asarray(charges_level, dtype=float); ok = np.isfinite(c)
    ob, op, ou = np.full(len(c), np.nan), np.full(len(c), np.nan), np.full(len(c), np.nan)
    if not ok.any():
        return ob, op, ou
    uc, inv = np.unique(c[ok], return_inverse=True)
    base = qbp + width; step = max(1, 2_000_000 // max(len(qbp), 1)); best = np.empty(len(uc), dtype=np.int64)
    for s in range(0, len(uc), step):
        best[s:s + step] = np.argmax((base[None, :] + uc[s:s + step, None]) * p[None, :], axis=1)
    bi = best[inv]
    ob[ok], op[ok], ou[ok] = qbp[bi], p[bi], (base[bi] + c[ok]) * p[bi]
    return ob, op, ou


def bt_opto(m, opt_bp) -> np.ndarray:
    return np.asarray(m, float) + np.asarray(opt_bp, float) / 100.0


def bt_final(opto, rows: np.ndarray | None = None) -> np.ndarray:
    """The street quote: the optimal yield plus the request's logged loading, as production applied it."""
    return np.asarray(opto, float) + (_adj if rows is None else _adj[rows])


def bt_win_fast(rows: np.ndarray, y: np.ndarray) -> np.ndarray:
    """The outcome rule of lab 14 [3] on a subset of rows: a quote richer than the winning print wins, a quote cheaper than our cover loses."""
    d = 100.0 * (_thr[rows] - y)
    return np.where(_grp[rows] == 'lost, printed', d > BT_EPS_BP, d >= -BT_EPS_BP) & np.isfinite(d)


def requote(rows: np.ndarray, charges: np.ndarray, width: float = 0.0) -> np.ndarray:
    """Production optimizer on the rows' own curves with the given charge levels; optimal yields (NaN where the key is not in the file)."""
    out = np.full(len(rows), np.nan); ch = np.asarray(charges, float)
    for (f, k), ii in pd.DataFrame({'f': _files[rows], 'k': _keys[rows]}).groupby(['f', 'k'], sort=False).indices.items():
        cur = BT_CURVES.get(f, {}).get(k)
        if cur is not None:
            out[ii] = bt_optimize(cur[0], cur[1], ch[ii], width)[0]
    return bt_opto(_m_sig[rows], out)


_all = np.arange(N); _opt_rep = requote(_all, _ch)
_rep_bp = 100.0 * (_opt_rep - _m_sig) - _xlog
REPLAY_BP = float(np.nanmax(np.abs(_rep_bp))); REPLAY_SHARE = float(np.mean(np.abs(np.nan_to_num(_rep_bp, nan=1e9)) <= 0.25))
_y_rep = bt_final(_opt_rep); _w_rep = bt_win_fast(_all, _y_rep)
REPLAY_PASS = bool(REPLAY_SHARE >= 0.95 and (_w_rep == _wlog).all() and np.allclose(_y_rep, _y_van, atol=1e-9, equal_nan=True))
_y_van = _y_rep
print(f'handoff {META["spec"]} written {META["run_at"][:19]}: {N:,} requests {BT_MONTHS[0]}..{BT_MONTHS[-1]}, winnable {_winable.mean():.1%}, logged win rate {_wlog.mean():.2%}, '
      f'manual requests {BT["manual"].astype(bool).mean():.1%} | test months {TEST_M[0]}..{TEST_M[-1]} ({len(TEST_M)}): {_tm.sum():,} requests, {_tmw.sum():,} winnable')
print(f'exits: record = {EX_RECORD}, guard = {EX_GUARD}; markout coverage of winnable requests record {np.isfinite(_M_REC[_winable]).mean():.1%}, guard {np.isfinite(_M_GUARD[_winable]).mean():.1%}, '
      f'other side {np.isfinite(_M_OTHER[_winable]).mean():.1%}, hedge term {np.isfinite(_mmd5[_winable]).mean():.1%} | covariance on {len(_sig_days)} of {_pb_nd} days ({META.get("sigma_label")}, betas {META.get("gz_label")})')
print('signal coverage of winnable requests:', {SIG_NAMES.get(k, k): round(float(np.isfinite(v[_winable]).mean()), 3) for k, v in SIG.items()}, '| conditional scale', round(float(np.isfinite(_a_risk[_winable]).mean()), 3))
print(f'REPLAY CHECK: optimal quantile reproduced within 0.25 bp on {REPLAY_SHARE:.2%} (max |d| {REPLAY_BP:.3g} bp); every logged win and loss reproduced: {(_w_rep == _wlog).all()}; '
      f'lab vanilla pass {META.get("vanilla_pass")} -> {"PASS" if REPLAY_PASS else "FAIL"}')
assert REPLAY_PASS, 'production not reproduced: nothing below is comparable'
_hold_req = np.full(N, float(BT_HOLD))
if HOLD_TAB is not None and 'tenor_bucket' in BT and 'call_bucket' in BT:
    _hk = HOLD_TAB[HOLD_TAB['tenor_bucket'] != 'ALL'].set_index(['tenor_bucket', 'call_bucket'])['hold_bdays']
    _hr = pd.MultiIndex.from_arrays([BT['tenor_bucket'].astype(str), BT['call_bucket'].astype(str)]).map(_hk.to_dict()).to_numpy(float)
    _hall = float(HOLD_TAB.loc[HOLD_TAB['tenor_bucket'] == 'ALL', 'hold_bdays'].iloc[0]) if (HOLD_TAB['tenor_bucket'] == 'ALL').any() else float(BT_HOLD)
    _hold_req = np.where(np.isfinite(_hr), _hr, _hall); _hold_req = np.clip(_hold_req, 1.0, 60.0)
    print(f'realised holding by tenor x call bucket (17): median over requests {np.median(_hold_req[_winable]):.1f} business days (lab horizon {BT_HOLD}); the charge family scales variance by h_i / {BT_HOLD}')
else:
    print(f'no realised holding table in the handoff: the charge family uses the lab horizon h = {BT_HOLD}')
record('variants_handoff', spec=META['spec'], requests=int(N), test_months=TEST_M, replay_pass=REPLAY_PASS, replay_max_bp=REPLAY_BP, exit_record=EX_RECORD, exit_guard=EX_GUARD,
       markout_coverage_record=float(np.isfinite(_M_REC[_winable]).mean()), holding_median=float(np.median(_hold_req[_winable])), seconds=time.perf_counter() - _h_t0)

## 3. Machinery

Quoting by charge (`requote`, above) and by curve (`requote_curve`: a per-request fill curve on the key's grid, the same argmax), the levelling
width, the sequential book for the inventory charges, and one paired scorecard used by every pair. The scorecard values a fill at the record exit,
the guard, the common markout (new wins only) and the hedged record exit, and reports the frontier of the comparator's regime at the variant's win rate.

In [ ]:
CELL_T('3. Machinery [3]')


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))


def requote_curve(rows: np.ndarray, charges: np.ndarray, ratio: np.ndarray | None = None, shift: np.ndarray | None = None, curves: dict | None = None,
                  curve_of: np.ndarray | None = None) -> np.ndarray:
    """Production's argmax with a per-request fill curve: p(q x ratio_i) (stretch), sigmoid(logit p + shift_i) (logit tilt), or curves[(f, k, curve_of_i)] (a refit).
    ratio = None and shift = None and curves = None reproduces requote."""
    out = np.full(len(rows), np.nan); ch = np.asarray(charges, float)
    for (f, k), ii in pd.DataFrame({'f': _files[rows], 'k': _keys[rows]}).groupby(['f', 'k'], sort=False).indices.items():
        cur = BT_CURVES.get(f, {}).get(k)
        if cur is None:
            continue
        q, p = cur
        for s in range(0, len(ii), CFG.chunk):
            jj = ii[s:s + CFG.chunk]; cj = ch[jj]; ok = np.isfinite(cj)
            if not ok.any():
                continue
            if ratio is not None:
                X = q[None, :] * ratio[jj][:, None]; P = np.interp(X.ravel(), q, p).reshape(X.shape)
            elif shift is not None:
                P = _sigmoid(_logit(p)[None, :] + shift[jj][:, None])
            elif curves is not None:
                P = np.vstack([curves.get((f, k, int(t)), p) for t in curve_of[jj]])
            else:
                P = np.broadcast_to(p[None, :], (len(jj), len(q)))
            U = (q[None, :] + np.where(ok, cj, 0.0)[:, None]) * P; bi = np.argmax(U, axis=1)
            o = q[bi].astype(float); o[~ok] = np.nan; out[jj] = o
    return bt_opto(_m_sig[rows], out)


def p_at(rows: np.ndarray, x: np.ndarray, ratio: np.ndarray | None = None, shift: np.ndarray | None = None, curves: dict | None = None, curve_of: np.ndarray | None = None) -> np.ndarray:
    """The (reshaped) fill probability at concession x for each row; the validation of a curve form at the logged concession."""
    out = np.full(len(rows), np.nan)
    for (f, k), ii in pd.DataFrame({'f': _files[rows], 'k': _keys[rows]}).groupby(['f', 'k'], sort=False).indices.items():
        cur = BT_CURVES.get(f, {}).get(k)
        if cur is None:
            continue
        q, p = cur; xi = x[ii]
        if ratio is not None:
            out[ii] = np.interp(xi * ratio[ii], q, p)
        elif shift is not None:
            out[ii] = _sigmoid(_logit(np.interp(xi, q, p)) + shift[ii])
        elif curves is not None:
            for t in np.unique(curve_of[ii]):
                sel = curve_of[ii] == t; out[ii[sel]] = np.interp(xi[sel], q, curves.get((f, k, int(t)), p))
        else:
            out[ii] = np.interp(xi, q, p)
    return out


def level_to(y_v: np.ndarray, y_b: np.ndarray, rows: np.ndarray) -> tuple[np.ndarray, dict]:
    """A constant width per month (bp, positive = cheaper) added to y_v so its win rate on the month's paired rows equals y_b's: the width nearest zero that does it, by bisection."""
    y = y_v.copy(); widths = {}
    for m in TEST_M:
        r = rows[_month[rows] == m]
        if not len(r):
            continue
        target = float(bt_win_fast(r, y_b[r]).mean())

        def wr(w: float) -> float:
            return float(bt_win_fast(r, y_v[r] + w / 100.0).mean())

        w0 = wr(0.0)
        if abs(w0 - target) <= 1e-12:
            w = 0.0                                          # already at the comparator's win rate: no width
        elif w0 > target:                                    # wins more: the smallest cheapening that brings it down to the target
            lo, hi = 0.0, CFG.level_range_bp
            if wr(hi) > target:
                w = hi
            else:
                for _ in range(CFG.level_iters):
                    mid = 0.5 * (lo + hi); lo, hi = (mid, hi) if wr(mid) > target else (lo, mid)
                w = hi
        else:                                                # wins less: the smallest richening that brings it up to the target
            lo, hi = -CFG.level_range_bp, 0.0
            if wr(lo) < target:
                w = lo
            else:
                for _ in range(CFG.level_iters):
                    mid = 0.5 * (lo + hi); lo, hi = (lo, mid) if wr(mid) < target else (mid, hi)
                w = lo
        y[r] = y_v[r] + w / 100.0; widths[m] = w
    return y, widths


def book_sd(win: np.ndarray) -> dict:
    """Ex-ante daily sd of the policy's own book on the test days (fills held BT_HOLD days), gross DV01: lab bt_book_g without the realised part."""
    w = win & (_pb_u > 0); di = _pb_di[w] + 1
    F = np.column_stack([np.bincount(di, weights=EXPO[w, c], minlength=_pb_nd + 1) for c in range(_D)]); I = np.bincount(di, weights=V_IDIO[w], minlength=_pb_nd + 1); U = np.bincount(di, weights=_pb_u[w], minlength=_pb_nd + 1)
    F, I, U = F.cumsum(0), I.cumsum(0), U.cumsum(0); j = np.arange(_pb_nd); lo = np.maximum(0, j - (BT_HOLD - 1)); E, Iv, Uv = F[j] - F[lo], I[j] - I[lo], U[j] - U[lo]
    sd = np.array([np.sqrt(max(E[k] @ SIGMA[k] @ E[k], 0.0) + Iv[k]) if k in SIGMA else np.nan for k in j])
    d = _tdays & np.isfinite(sd)
    return {'sd': float(np.mean(sd[d])) if d.any() else np.nan, 'dv01': float(np.mean(Uv[_tdays])) if _tdays.any() else np.nan}


def s_book(rows: np.ndarray, j: int, E: np.ndarray, expo: np.ndarray) -> np.ndarray:
    """Marginal risk of each fill against book exposure E on day j (lab 14e bt_marginal_g): BT_HOLD (2 e'S E + e'S e + v) / (2 u), bp per unit gamma."""
    S = SIGMA[j]; e = expo[rows]
    return BT_HOLD * (2.0 * e @ (S @ E) + np.einsum('ij,jk,ik->i', e, S, e) + V_IDIO[rows]) / (2.0 * _pb_u[rows])


def book_replay(c0: np.ndarray, o0: np.ndarray, kind: str, gamma: float, extra_bp: float = 0.0, curve: dict | None = None) -> np.ndarray:
    """Sequential by business day from the first day with a covariance: the policy's own start-of-day book (fills of the last BT_HOLD days) sets the
    inventory charge of each test-month request; earlier days quote the base and only build the book. kind in {'own', 'book', 'hedged'}; curve = a
    requote_curve keyword set for the combined variant."""
    opto = o0.copy(); F = np.zeros((_pb_nd, _D)); Q = np.zeros((_pb_nd, _n_cusip)); expo = EXPO_H if kind == 'hedged' else EXPO
    vol2 = VOL['total'] ** 2 * (_hold_req / BT_HOLD)
    for j in range(_pb_nd):
        rows = _pb_rows_day.get(j, np.zeros(0, int)); act = rows[np.isfinite(o0[rows]) & (_pb_u[rows] > 0)]
        if not len(act):
            continue
        if j in SIGMA and _tdays[j]:
            lo = max(0, j - (BT_HOLD - 1))
            if kind == 'own':
                Qi = Q[lo:j].sum(0)[_ci[act]]; s = vol2[act] * (Qi + _pb_u[act] / 2.0)
            else:
                s = s_book(act, j, F[lo:j].sum(0), expo)
            c_ = c0[act] - extra_bp - gamma * s
            opto[act] = requote(act, c_) if curve is None else requote_curve(act, c_, **{k: (v[act] if isinstance(v, np.ndarray) else v) for k, v in curve.items()})
        w = bt_win_fast(act, opto[act] + _adj[act])
        F[j] = expo[act][w].sum(0); np.add.at(Q[j], _ci[act][w], _pb_u[act][w])
    return opto


def score_pair(name: str, y_v: np.ndarray, y_b: np.ndarray, frontier: pd.DataFrame | None, family: str = '', setting: str = '', regime: str = '', pair: str = '', mode: str = 'raw',
               widths: dict | None = None, bk_base: dict | None = None) -> dict:
    """The paired scorecard: variant quote y_v against comparator y_b on the test-month rows where both exist."""
    rows = np.flatnonzero(_tmw & np.isfinite(y_v) & np.isfinite(y_b)); yv, yb = y_v[rows], y_b[rows]
    wv, wb = bt_win_fast(rows, yv), bt_win_fast(rows, yb); g = _grp[rows]; dv, db = 100.0 * (_thr[rows] - yv), 100.0 * (_thr[rows] - yb)
    px_v, px_b = 100.0 * (yv - _y_tr[rows]), 100.0 * (yb - _y_tr[rows]); kept = wv & wb; new = wv & ~wb; gone = wb & ~wv; keptc = kept & (g == 'won, cover')
    res = {'variant': name, 'family': family, 'setting': setting, 'regime': regime, 'pair': pair, 'mode': mode, 'months': len(set(_month[rows])), 'requests': len(rows),
           'win rate': float(wv.mean()), 'win rate, base': float(wb.mean()), 'wins kept': float(kept.sum() / max(wb.sum(), 1)), 'new wins': int(new.sum()), 'wins given up': int(gone.sum()),
           'bp left on table (won, cover)': float(dv[wv & (g == 'won, cover')].mean()) if (wv & (g == 'won, cover')).any() else np.nan,
           'bp left on table, base': float(db[wb & (g == 'won, cover')].mean()) if (wb & (g == 'won, cover')).any() else np.nan,
           'bp saved on kept cover wins': float((db - dv)[keptc].mean()) if keptc.any() else np.nan,
           'crossing on new wins (bp)': float(-dv[new & (g == 'lost, printed')].mean()) if (new & (g == 'lost, printed')).any() else np.nan,
           'levelling width (bp)': float(np.mean(list(widths.values()))) if widths else 0.0}
    vals = {'record': (_M_REC[rows], _M_REC[rows]), 'guard': (_M_GUARD[rows], _M_GUARD[rows]), 'common markout': (np.where(new, _cm_rec[rows], _M_REC[rows]), _M_REC[rows]),
            'hedged': (_M_REC[rows] + _mmd5[rows] - CFG.hedge_cost_bp, _M_REC[rows] + _mmd5[rows] - CFG.hedge_cost_bp), 'other side': (_M_OTHER[rows], _M_OTHER[rows])}
    for key, (Mv, Mb) in vals.items():
        pv = np.where(wv, px_v + Mv, 0.0); pb = np.where(wb, px_b + Mb, 0.0); S = np.isfinite(pv) & np.isfinite(pb) & ~((wv & ~np.isfinite(Mv)) | (wb & ~np.isfinite(Mb)))
        d = np.where(S, pv - pb, np.nan); wk = _group_mean(d, _wk_code[rows], len(_wk_levels)); mm = _group_mean(d, _mo_code[rows], len(_mo_levels))
        res[f'delta | {key} (bp)'] = float(np.nanmean(d)) if S.any() else np.nan; res[f'NW t | {key}'] = nw_t(wk); res[f'months > 0 | {key}'] = int(np.nansum(mm > 0))
        if key == 'record':
            res['P&L sample share'] = float(S.mean()); res['P&L per request (bp)'] = float(np.nanmean(pv[S])); res['P&L per request, base (bp)'] = float(np.nanmean(pb[S]))
            res['delta: common fills price'] = float(np.sum((px_v - px_b)[kept & S]) / max(S.sum(), 1)); res['delta: new wins'] = float(np.sum(pv[new & S]) / max(S.sum(), 1))
            res['delta: wins given up'] = float(-np.sum(pb[gone & S]) / max(S.sum(), 1))
            res['delta $ per month ($k)'] = float((np.nansum(pv[S] * _usd[rows][S]) - np.nansum(pb[S] * _usd[rows][S])) / 1e3 / max(res['months'], 1))
    if frontier is not None and len(frontier):
        feas = frontier[frontier['win rate'] >= res['win rate'] - 1e-12]
        res['above width frontier (bp)'] = res['P&L per request (bp)'] - (float(feas['P&L per request (bp)'].max()) if len(feas) else np.nan)
    else:
        res['above width frontier (bp)'] = np.nan
    full_v = np.zeros(N, bool); full_v[rows] = wv; bk = book_sd(full_v)
    if bk_base is None:
        full_b = np.zeros(N, bool); full_b[rows] = wb; bk_base = book_sd(full_b)
    bk0 = bk_base
    res['ex-ante daily sd ($k)'] = bk['sd'] / 1e3; res['ex-ante sd vs base'] = bk['sd'] / bk0['sd'] - 1.0 if bk0['sd'] and np.isfinite(bk0['sd']) else np.nan; res['gross DV01 ($k/bp)'] = bk['dv01'] / 1e3
    return res


def label4(gate: bool, est: float, t: float, flips: bool) -> str:
    if gate:
        return 'accepted under the specified gate'
    if pd.notna(est) and pd.notna(t) and est > 0 and t >= 2.0 and not flips:
        return 'positive evidence, gate not met'
    if pd.notna(t) and ((est < 0 and t <= -2.0) or flips):
        return 'negative evidence'
    return 'inconclusive'


def wf_pick(settings: dict[str, tuple[np.ndarray, np.ndarray]], order: list[str]) -> tuple[np.ndarray, np.ndarray, list[dict]]:
    """Walk-forward pick among settings per test month by the hedged weekly $ Sharpe of the manual-kept quote on the earlier test months; the first month takes order[0]."""
    def sharpe(y: np.ndarray, months: list[str]) -> float:
        r = np.flatnonzero(_tmw & np.isfinite(y) & np.isin(_month, months))
        if not len(r):
            return np.nan
        w = bt_win_fast(r, y[r]); pnl = np.where(w, (100.0 * (y[r] - _y_tr[r]) + _M_REC[r] + _mmd5[r] - CFG.hedge_cost_bp) * _usd[r], 0.0); ok = np.isfinite(pnl)
        wk = np.bincount(_wk_code[r][ok], weights=pnl[ok], minlength=len(_wk_levels)); has = np.bincount(_wk_code[r][ok], minlength=len(_wk_levels)) > 0; wk = wk[has]
        return float(wk.mean() / wk.std(ddof=1)) if len(wk) > 2 and wk.std(ddof=1) > 0 else np.nan

    yM, y0, rows = np.full(N, np.nan), np.full(N, np.nan), []
    for i, m in enumerate(TEST_M):
        earlier = TEST_M[:i]
        if earlier:
            sc = {k: sharpe(v[0], earlier) for k, v in settings.items()}; best = max(sc, key=lambda k: (np.nan_to_num(sc[k], nan=-np.inf)))
        else:
            sc, best = {}, order[0]
        te = _month == m; yM[te] = settings[best][0][te]; y0[te] = settings[best][1][te]; rows.append({'test month': m, 'picked': best, **{f'train Sharpe {k}': v for k, v in sc.items()}})
    return yM, y0, rows


VARIANTS: dict[str, dict] = {}   # name -> {'family', 'setting', 'yM': quote with manual kept, 'y0': quote with manual removed, ...}
print('machinery ready: requote by charge and by curve, levelling, the sequential book (own CUSIP, book, hedged), the paired scorecard with five valuations')

## 4. References: production with and without the manual charge; pair D

Production quoted by the optimizer twice: with its logged charge `c = -(manual + liquidity + risk)` and with `c_base = -(liquidity + risk)`.
The width grids of both regimes give the frontiers. Pair D is the manual charge's own scorecard, raw and levelled; the levelled row is the
"flat" control of lab 14c: the manual charge replaced by the constant width that keeps production's win rate.

In [ ]:
CELL_T('4. References and pair D [4]')
_r_t0 = time.perf_counter()
_tm_rows = np.flatnonzero(_tmw)
Y_PM = _y_van.copy()                                             # production, manual kept (the replay)
Y_P0 = np.full(N, np.nan); Y_P0[_tm_rows] = bt_final(requote(_tm_rows, _c_base[_tm_rows]), _tm_rows)    # production, manual removed


def _width_grid(c: np.ndarray) -> pd.DataFrame:
    rows_ = []
    for w in CFG.width_grid:
        y = np.full(N, np.nan); y[_tm_rows] = bt_final(requote(_tm_rows, c[_tm_rows], float(w)), _tm_rows)
        wv = bt_win_fast(_tm_rows, y[_tm_rows]); pv = np.where(wv, 100.0 * (y[_tm_rows] - _y_tr[_tm_rows]) + _M_REC[_tm_rows], 0.0)
        rows_.append({'width': int(w), 'win rate': float(wv.mean()), 'P&L per request (bp)': float(np.nanmean(pv))})
    return pd.DataFrame(rows_)


FRONTIER = {'M': cached('var_frontier_M', lambda: _width_grid(_ch), deps=[frame_fingerprint(BT, ['charges_level', 'threshold_y']), TEST_M], code=[requote, bt_optimize], cfg=('width_grid',)),
            '0': cached('var_frontier_0', lambda: _width_grid(_c_base), deps=[frame_fingerprint(BT, ['liquiditycharge', 'riskcharge', 'threshold_y']), TEST_M], code=[requote, bt_optimize], cfg=('width_grid',))}
print('Width frontiers on the test-month winnable requests (P&L at the record exit by constant width, both regimes):')
display(pd.concat({'manual kept': FRONTIER['M'].set_index('width'), 'manual removed': FRONTIER['0'].set_index('width')}, axis=1).round(4))
_wbM, _wb0 = np.zeros(N, bool), np.zeros(N, bool); _wbM[_tm_rows] = bt_win_fast(_tm_rows, Y_PM[_tm_rows]); _wb0[_tm_rows] = bt_win_fast(_tm_rows, Y_P0[_tm_rows])
BK_BASE = {'M': book_sd(_wbM), '0': book_sd(_wb0)}   # the comparators' own books, computed once
_d_raw = score_pair('production, manual removed', Y_P0, Y_PM, FRONTIER['M'], family='reference', setting='manual removed', regime='0', pair='D', mode='raw', bk_base=BK_BASE['M'])
_y_lev, _wd = level_to(Y_P0, Y_PM, _tm_rows)
_d_lev = score_pair('production, manual removed', _y_lev, Y_PM, None, family='reference', setting='manual removed', regime='0', pair='D', mode='levelled', widths=_wd, bk_base=BK_BASE['M'])
PAIR_D = pd.DataFrame([_d_raw, _d_lev]).set_index('mode')
_show_d = ['win rate', 'win rate, base', 'wins kept', 'new wins', 'wins given up', 'bp left on table (won, cover)', 'bp left on table, base', 'bp saved on kept cover wins', 'crossing on new wins (bp)',
           'levelling width (bp)', 'delta | record (bp)', 'NW t | record', 'delta | guard (bp)', 'delta | common markout (bp)', 'delta | hedged (bp)', 'delta | other side (bp)',
           'delta $ per month ($k)', 'above width frontier (bp)', 'ex-ante daily sd ($k)', 'ex-ante sd vs base', 'gross DV01 ($k/bp)']
_man_m = pd.Series(_man[_tmw]).groupby(_month[_tmw]).agg(['mean', 'median']); _man_m.columns = ['manual charge, mean (bp)', 'median (bp)']
_wr_m = pd.DataFrame({'win rate, manual kept': pd.Series(bt_win_fast(_tm_rows, Y_PM[_tm_rows])).groupby(_month[_tm_rows]).mean(),
                      'win rate, manual removed': pd.Series(bt_win_fast(_tm_rows, Y_P0[_tm_rows])).groupby(_month[_tm_rows]).mean()})
print('The manual charge by test month, and production\'s win rate with and without it:'); display(_man_m.join(_wr_m).round(4))
print('PAIR D (reference): production without the manual charge against production; levelled = the constant width per month that returns to production\'s win rate:')
display(PAIR_D[_show_d].T.round(4))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for k_, col in (('M', '#4C72B0'), ('0', '#C44E52')):
    ax[0].plot(FRONTIER[k_]['win rate'], FRONTIER[k_]['P&L per request (bp)'], 'o-', color=col, label='manual kept' if k_ == 'M' else 'manual removed')
ax[0].set_xlabel('win rate'); ax[0].set_ylabel('P&L per request, record exit (bp)'); ax[0].set_title('Width frontiers of the two regimes', fontsize=10); ax[0].legend(fontsize=8)
ax[1].bar(_wr_m.index, _wr_m['win rate, manual kept'], width=0.4, color='#4C72B0', label='manual kept'); ax[1].bar(np.arange(len(_wr_m)) + 0.4, _wr_m['win rate, manual removed'], width=0.4, color='#C44E52', label='manual removed')
ax[1].set_title('Production win rate by month, with and without the manual charge', fontsize=10); ax[1].legend(fontsize=8); ax[1].tick_params(axis='x', labelsize=8)
savefig('v4_references')
record('variants_references', frontier_M=flat_records(FRONTIER['M']), frontier_0=flat_records(FRONTIER['0']), pair_D=flat_records(PAIR_D), manual_by_month=flat_records(_man_m.join(_wr_m)),
       seconds=time.perf_counter() - _r_t0)
print(f'[4] {time.perf_counter() - _r_t0:.0f}s')

## 5. The fill curve reshaped by volatility

Production's curve per key, reshaped per request by the bond's volatility: a horizontal stretch `p(x (sigma_ref / sigma_i)^a)`, a logit tilt
`logit p + kappa z_i`, and the desk's own curve by volatility tercile shrunk to production's. `sigma_ref` is the key's median sigma on the earlier
months and `z_i` the standardised log sigma, both walk-forward. Four sigmas: systematic, idiosyncratic and total from 14d, and the lab's conditional
scale. Every form is validated at the logged concession first (held-out log-likelihood against production's curve, calibration by sigma decile),
then quoted in both regimes. The validation is reported, not gating: the replay decides.

In [ ]:
CELL_T('5. The fill curve reshaped by volatility [5]')
_c_t0 = time.perf_counter()
SIGMAS = {'systematic': VOL.get('rate + factor'), 'idiosyncratic': VOL.get('idiosyncratic'), 'total': VOL.get('total'), 'conditional scale': _a_risk}
SIGMAS = {k: v for k, v in SIGMAS.items() if v is not None and np.isfinite(v[_winable]).mean() > 0.2}
_x_won = _wlog; _X = 100.0 * (_thr - _m_sig - _adj); _won_g, _lost_g = np.isin(_grp, ['won, cover', 'won, no cover']), _grp == 'lost, printed'


def _ref_by_key(sig: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Per request: the key's median sigma, the pooled mean and sd of log sigma, all on the months before the request's month (winnable requests with sigma)."""
    ref, mu, sd = np.full(N, np.nan), np.full(N, np.nan), np.full(N, np.nan); ls = np.log(np.where(sig > 0, sig, np.nan))
    for m in BT_MONTHS[1:]:
        tr = (_month < m) & _winable & np.isfinite(ls); te = _month == m
        if tr.sum() < 100:
            continue
        med = pd.Series(sig[tr]).groupby(_keys[tr]).median(); ref[te] = pd.Series(_keys[te]).map(med).to_numpy(float)
        ref[te & ~np.isfinite(ref)] = float(np.median(sig[tr])); mu[te], sd[te] = float(np.mean(ls[tr])), float(np.std(ls[tr]))
    return ref, mu, sd


def _tercile_curves(sig: np.ndarray, m: str, n0: float, train: np.ndarray) -> tuple[dict, np.ndarray]:
    """Per (file, key, tercile): the desk's own curve on the training rows shrunk to production with pseudo-count n0; terciles of log sigma on the training rows."""
    ls = np.log(np.where(sig > 0, sig, np.nan)); q1, q2 = np.nanquantile(ls[train], [1 / 3, 2 / 3]); terc = np.where(ls <= q1, 0, np.where(ls <= q2, 1, 2)); terc[~np.isfinite(ls)] = 1
    curves = {}
    if n0 == np.inf:
        return curves, terc
    for (f, k, t), ii in pd.DataFrame({'f': _files[train], 'k': _keys[train], 't': terc[train]}).groupby(['f', 'k', 't'], sort=False).indices.items():
        cur = BT_CURVES.get(f, {}).get(k)
        if cur is None:
            continue
        q, p = cur; rows = np.flatnonzero(train)[ii]; xw, xl = np.sort(_X[rows][_won_g[rows]]), np.sort(_X[rows][_lost_g[rows]]); n = len(rows)
        emp = ((len(xw) - np.searchsorted(xw, q, 'left')) + (len(xl) - np.searchsorted(xl, q, 'right'))) / max(n, 1)
        curves[(f, k, int(t))] = (n * emp + (0.0 if n0 == 0 else n0) * p) / (n + (0.0 if n0 == 0 else n0))
    return curves, terc


def _ll(p: np.ndarray, won: np.ndarray) -> np.ndarray:
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.where(won, np.log(p), np.log(1 - p))


CURVE_FORMS = [('stretch', a) for a in CFG.stretch_exponents] + [('logit', k) for k in CFG.logit_kappas] + [('tercile', None)]


def _curve_family() -> dict:
    out, val_rows, cal_rows = {}, [], []
    p_prod = p_at(_all, _xlog); ll0 = _ll(p_prod, _x_won)
    for sname, sig in SIGMAS.items():
        ref, mu, sd = _ref_by_key(sig); z = np.clip((np.log(np.where(sig > 0, sig, np.nan)) - mu) / np.where(sd > 0, sd, np.nan), -3, 3); z = np.nan_to_num(z)
        for form, par in CURVE_FORMS:
            t0 = time.perf_counter(); nm = f'{form} {par:g} | {sname}' if par is not None else f'{form} | {sname}'
            yM, y0 = np.full(N, np.nan), np.full(N, np.nan); p_hat = np.full(N, np.nan); dec = np.full(N, np.nan)
            if form == 'stretch':
                ratio = np.where(np.isfinite(ref) & (sig > 0), (ref / np.where(sig > 0, sig, np.nan)) ** par, 1.0); kw = {'ratio': ratio}
                sel = _tm_rows; yM[sel] = requote_curve(sel, _ch[sel], ratio=ratio[sel]); y0[sel] = requote_curve(sel, _c_base[sel], ratio=ratio[sel])
                eval_rows = np.flatnonzero(np.isfinite(ref)); p_hat[eval_rows] = p_at(eval_rows, _xlog[eval_rows], ratio=ratio[eval_rows])
            elif form == 'logit':
                shift = par * z; kw = {'shift': shift}
                sel = _tm_rows; yM[sel] = requote_curve(sel, _ch[sel], shift=shift[sel]); y0[sel] = requote_curve(sel, _c_base[sel], shift=shift[sel])
                eval_rows = np.flatnonzero(np.isfinite(ref)); p_hat[eval_rows] = p_at(eval_rows, _xlog[eval_rows], shift=shift[eval_rows])
            else:   # tercile refit: n0 per month by the previous month's log-likelihood, curves on the months before
                n0_rows = []; kw = None
                for i, m in enumerate(BT_MONTHS[1:], start=1):
                    train = (_month < m) & np.isfinite(_X) & np.isfinite(_xlog); te = _month == m
                    if train.sum() < 100:
                        continue
                    prev = BT_MONTHS[i - 1]; tr_prev = (_month < prev) & np.isfinite(_X) & np.isfinite(_xlog); v_prev = (_month == prev) & np.isfinite(_xlog)
                    lls = {}
                    for n0 in CFG.n0_grid:
                        if tr_prev.sum() < 100:
                            lls[n0] = -np.inf if n0 != np.inf else 0.0; continue
                        cv, tc = _tercile_curves(sig, prev, n0, tr_prev); vr = np.flatnonzero(v_prev); lls[n0] = float(np.sum(_ll(p_at(vr, _xlog[vr], curves=cv, curve_of=tc), _x_won[vr])))
                    n0 = max(lls, key=lls.get); cv, tc = _tercile_curves(sig, m, n0, train); ter = np.flatnonzero(te); p_hat[ter] = p_at(ter, _xlog[ter], curves=cv, curve_of=tc)
                    if m in TEST_M:
                        tw = ter[_winable[ter]]; yM[tw] = requote_curve(tw, _ch[tw], curves=cv, curve_of=tc); y0[tw] = requote_curve(tw, _c_base[tw], curves=cv, curve_of=tc)
                    n0_rows.append({'month': m, 'n0': n0})
                eval_rows = np.flatnonzero(np.isfinite(p_hat))
            ev = eval_rows[np.isfinite(_xlog[eval_rows]) & np.isfinite(p_hat[eval_rows]) & (_month[eval_rows] >= BT_MONTHS[1])]; gain = _ll(p_hat[ev], _x_won[ev]) - ll0[ev]
            mm = pd.Series(gain).groupby(_month[ev]).mean(); ok = np.isfinite(sig[ev]) & (sig[ev] > 0)
            dec[ev[ok]] = pd.qcut(pd.Series(sig[ev][ok]).rank(method='first'), 10, labels=False).to_numpy()
            cal = pd.DataFrame({'decile': dec[ev], 'p_hat': p_hat[ev], 'p_prod': p_prod[ev], 'won': _x_won[ev].astype(float)}).dropna().groupby('decile').mean()
            val_rows.append({'variant': nm, 'sigma': sname, 'form': form, 'requests': int(len(ev)), 'LL gain per request vs production': float(gain.mean()), 'months with LL gain > 0': f'{int((mm > 0).sum())}/{len(mm)}',
                             'calibration: max |p_hat - won| by sigma decile': float((cal['p_hat'] - cal['won']).abs().max()) if len(cal) else np.nan,
                             'calibration: max |p_prod - won| by sigma decile': float((cal['p_prod'] - cal['won']).abs().max()) if len(cal) else np.nan,
                             'quoted test requests': int(np.isfinite(yM).sum()), 'seconds': time.perf_counter() - t0})
            cal_rows += [{'variant': nm, 'decile': int(d), **r.to_dict()} for d, r in cal.iterrows()]
            out[nm] = {'family': 'fill curve', 'setting': nm, 'yM': bt_final(yM), 'y0': bt_final(y0), 'kw': kw, 'sigma': sname}
            print(f'  {nm}: LL gain {val_rows[-1]["LL gain per request vs production"]:+.5f} per request, {val_rows[-1]["months with LL gain > 0"]} months | {time.perf_counter() - t0:.0f}s')
    return {'variants': out, 'validation': pd.DataFrame(val_rows).set_index('variant'), 'calibration': pd.DataFrame(cal_rows)}


_cf = cached('var_curves', _curve_family, deps=[frame_fingerprint(BT, ['charges_level', 'threshold_y', 'optimalQuantileBp']), TEST_M, {k: float(np.nansum(v)) for k, v in SIGMAS.items()}],
             code=[requote_curve, p_at, _ref_by_key, _tercile_curves, bt_optimize], cfg=('stretch_exponents', 'logit_kappas', 'n0_grid', 'chunk'))
CURVE_VAL, CURVE_CAL = _cf['validation'], _cf['calibration']
for k_, v_ in _cf['variants'].items():
    VARIANTS[k_] = v_
print('Fill-curve validation at the logged concession (months after the first; LL gain > 0 = the reshaped curve predicts the logged outcome better than production\'s):')
display(CURVE_VAL.drop(columns=['seconds']).round(5))
print('Calibration by sigma decile of the best form (predicted fill probability at the logged concession against the logged win share):')
_best_form = CURVE_VAL['LL gain per request vs production'].idxmax() if len(CURVE_VAL) else None
if _best_form is not None:
    display(CURVE_CAL[CURVE_CAL['variant'] == _best_form].set_index('decile')[['p_hat', 'p_prod', 'won']].round(4).T)
print(f'[5] {time.perf_counter() - _c_t0:.0f}s')
record('variants_curves', validation=flat_records(CURVE_VAL), calibration=flat_records(CURVE_CAL), settings=list(_cf['variants']), seconds=time.perf_counter() - _c_t0)

## 6. Charges in the Avellaneda-Stoikov form

`delta_bid = half-spread + inventory x gamma sigma^2 (T - t)`: production already holds the half-spread's intensity term in `argmax (x + c) p(x)`;
the two volatility terms are added to `c` and quoted by the same argmax. Half-spread `gamma sigma_h^2 u / 2`; own-CUSIP inventory
`gamma sigma_h^2 (Q_i + u_i / 2)` with `Q_i` the policy's own DV01 in the CUSIP over the last h days; book inventory `gamma s_i` (lab 14e), full and
with the rate block hedged at the hedge cost; combined = the curve form with the best held-out log-likelihood plus the book charge. `gamma_k = k /
median s` on production's book in the first test month, so `k` is the median extra charge in bp; `k` picked walk-forward by the hedged weekly dollar
Sharpe. The horizon of a request is the realised holding of its bucket (17): `sigma_h^2 = sigma_5^2 h_i / 5`.

In [ ]:
CELL_T('6. Charges in the Avellaneda-Stoikov form [6]')
_g_t0 = time.perf_counter()
_vol2 = VOL['total'] ** 2 * (_hold_req / BT_HOLD)
_m1 = TEST_M[0]; _r1 = np.flatnonzero((_month == _m1) & _winable & (_pb_u > 0))
_w_van = np.zeros(N, bool); _w_van[_tm_rows] = bt_win_fast(_tm_rows, Y_PM[_tm_rows]); _w_van_all = bt_win_fast(_all, Y_PM)


def _s_on_vanilla(kind: str) -> np.ndarray:
    """s on production's own book (vanilla wins) for the first test month's requests: the scale of gamma."""
    if kind == 'half':
        return _vol2[_r1] * _pb_u[_r1] / 2.0
    F = np.zeros((_pb_nd, _D)); Q = np.zeros((_pb_nd, _n_cusip)); expo = EXPO_H if kind == 'hedged' else EXPO; s = np.full(len(_r1), np.nan); pos = pd.Series(np.arange(len(_r1)), index=_r1)
    for j in range(_pb_nd):
        rows = _pb_rows_day.get(j, np.zeros(0, int)); act = rows[(_pb_u[rows] > 0)]
        if not len(act):
            continue
        if _pb_month[j] == _m1 and j in SIGMA:
            lo = max(0, j - (BT_HOLD - 1)); a1 = act[np.isin(act, _r1)]
            if len(a1):
                s[pos[a1].to_numpy()] = (_vol2[a1] * (Q[lo:j].sum(0)[_ci[a1]] + _pb_u[a1] / 2.0)) if kind == 'own' else s_book(a1, j, F[lo:j].sum(0), expo)
        w = _w_van_all[act]; F[j] = expo[act][w].sum(0); np.add.at(Q[j], _ci[act][w], _pb_u[act][w])
        if _pb_month[j] > _m1:
            break
    return s


S_MED = {k: float(np.nanmedian(_s_on_vanilla(k))) for k in ('half', 'own', 'book', 'hedged')}
print('gamma scale: median s on production\'s book in', _m1, {k: f'{v:.3g}' for k, v in S_MED.items()}, '| gamma_k = k / median s, k in', CFG.k_grid)
_cv_par = CURVE_VAL[CURVE_VAL['form'] != 'tercile'] if len(CURVE_VAL) else CURVE_VAL   # the tercile refit's curves are per month and are not carried into the sequential replay
_best_curve = _cv_par['LL gain per request vs production'].idxmax() if len(_cv_par) else None
print(f'combined variant uses the parametric curve form with the best held-out log-likelihood: {_best_curve}')


def _charge_family() -> dict:
    out = {}
    o_M = np.full(N, np.nan); o_M[_tm_rows] = requote(_tm_rows, _ch[_tm_rows]); o_0 = np.full(N, np.nan); o_0[_tm_rows] = requote(_tm_rows, _c_base[_tm_rows])
    o_M_all = requote(_all, _ch); o_0_all = o_M_all.copy(); o_0_all[_tm_rows] = o_0[_tm_rows]   # outside the test months the book is built from production's quotes
    for kind in ('half', 'own', 'book', 'hedged', 'combined'):
        if kind == 'combined' and _best_curve is None:
            continue
        for k in CFG.k_grid:
            t0 = time.perf_counter(); g = k / S_MED['book' if kind == 'combined' else kind]; nm = f'{kind} | k={k}'
            if kind == 'half':
                s = _vol2 * _pb_u / 2.0; yM, y0 = np.full(N, np.nan), np.full(N, np.nan)
                yM[_tm_rows] = requote(_tm_rows, _ch[_tm_rows] - g * s[_tm_rows]); y0[_tm_rows] = requote(_tm_rows, _c_base[_tm_rows] - g * s[_tm_rows])
            else:
                extra = CFG.hedge_cost_bp if kind == 'hedged' else 0.0; curve = VARIANTS[_best_curve]['kw'] if kind == 'combined' else None
                yM = book_replay(_ch, o_M_all, 'book' if kind == 'combined' else kind, g, extra, curve); y0 = book_replay(_c_base, o_0_all, 'book' if kind == 'combined' else kind, g, extra, curve)
                yM[~_tmw] = np.nan; y0[~_tmw] = np.nan
            out[nm] = {'family': {'half': 'vol half-spread', 'own': 'own-CUSIP inventory', 'book': 'book inventory', 'hedged': 'book inventory, hedged', 'combined': 'curve + book inventory'}[kind],
                       'setting': f'k={k}', 'yM': bt_final(yM), 'y0': bt_final(y0)}
            print(f'  {nm}: {time.perf_counter() - t0:.0f}s')
    return out


_gf = cached('var_charges', _charge_family, deps=[frame_fingerprint(BT, ['charges_level', 'threshold_y']), TEST_M, float(np.nansum(_vol2)), float(np.nansum(EXPO)), S_MED, _best_curve, len(SIGMA)],
             code=[book_replay, s_book, requote, requote_curve, bt_optimize], cfg=('k_grid', 'hedge_cost_bp', 'chunk'))
for k_, v_ in _gf.items():
    VARIANTS[k_] = v_
CHARGE_PICKS = {}
for kind, fam in (('half', 'vol half-spread'), ('own', 'own-CUSIP inventory'), ('book', 'book inventory'), ('hedged', 'book inventory, hedged'), ('combined', 'curve + book inventory')):
    sets = {f'{kind} | k={k}': (VARIANTS[f'{kind} | k={k}']['yM'], VARIANTS[f'{kind} | k={k}']['y0']) for k in CFG.k_grid if f'{kind} | k={k}' in VARIANTS}
    if not sets:
        continue
    yM, y0, rows = wf_pick(sets, list(sets)); VARIANTS[f'{kind} | walk-forward k'] = {'family': fam, 'setting': 'walk-forward k', 'yM': yM, 'y0': y0}; CHARGE_PICKS[kind] = pd.DataFrame(rows).set_index('test month')
print('Walk-forward k per family (hedged weekly $ Sharpe of the manual-kept quote on the earlier test months; the first test month takes k = 1):')
_pk_show = []
for kind, t_ in CHARGE_PICKS.items():
    t_ = t_.copy(); t_.columns = [c.replace(f'train Sharpe {kind} | ', 'Sharpe ') for c in t_.columns]; t_['picked'] = t_['picked'].str.replace(f'{kind} | ', ''); t_.insert(0, 'family', kind); _pk_show.append(t_.reset_index())
display(pd.concat(_pk_show, ignore_index=True).set_index(['family', 'test month']).round(3))
record('variants_charges', s_median=S_MED, best_curve=_best_curve, picks={k: flat_records(v) for k, v in CHARGE_PICKS.items()}, seconds=time.perf_counter() - _g_t0)
print(f'[6] {time.perf_counter() - _g_t0:.0f}s')

## 7. The model signals as charges

Each signal of lab 14 [5], oriented so that positive means the bond is expected to richen, is standardised walk-forward and enters the charge as
`c + k z_i`: a bond expected to richen is charged less and bid for more aggressively, through the optimizer rather than as a shift of the fair
value. Zero where the signal is missing. Nothing here is gated by the model-level verdicts of lab Sections 7b to 12.

In [ ]:
CELL_T('7. The model signals as charges [7]')
_s_t0 = time.perf_counter()


def _z_walk(sig: np.ndarray) -> np.ndarray:
    z = np.zeros(N)
    for m in BT_MONTHS[1:]:
        tr = (_month < m) & _winable & np.isfinite(sig); te = (_month == m) & np.isfinite(sig)
        if tr.sum() < 100:
            continue
        xs = np.sort(sig[tr]); n = len(xs); F = np.interp(sig[te], xs, (np.arange(n) + 0.5) / n, left=0.5 / n, right=1 - 0.5 / n); z[te] = np.clip(stats.norm.ppf(F), -3, 3)
    return z


def _signal_family() -> dict:
    out = {}
    for s, sig in SIG.items():
        z = _z_walk(sig)
        for k in CFG.signal_k:
            t0 = time.perf_counter(); nm = f'{SIG_NAMES.get(s, s)} | k={k:g}'; yM, y0 = np.full(N, np.nan), np.full(N, np.nan)
            yM[_tm_rows] = requote(_tm_rows, _ch[_tm_rows] + k * z[_tm_rows]); y0[_tm_rows] = requote(_tm_rows, _c_base[_tm_rows] + k * z[_tm_rows])
            out[nm] = {'family': 'signal charge', 'setting': f'{SIG_NAMES.get(s, s)} k={k:g}', 'yM': bt_final(yM), 'y0': bt_final(y0), 'coverage': float(np.isfinite(sig[_tmw]).mean())}
            print(f'  {nm}: coverage of test winnable {out[nm]["coverage"]:.1%} | {time.perf_counter() - t0:.0f}s')
    return out


_sf = cached('var_signals', _signal_family, deps=[frame_fingerprint(BT, ['charges_level', 'threshold_y']), TEST_M, {k: float(np.nansum(v)) for k, v in SIG.items()}], code=[requote, _z_walk, bt_optimize], cfg=('signal_k',))
for k_, v_ in _sf.items():
    VARIANTS[k_] = v_
for s in SIG:
    sets = {f'{SIG_NAMES.get(s, s)} | k={k:g}': (VARIANTS[f'{SIG_NAMES.get(s, s)} | k={k:g}']['yM'], VARIANTS[f'{SIG_NAMES.get(s, s)} | k={k:g}']['y0']) for k in CFG.signal_k}
    yM, y0, _ = wf_pick(sets, list(sets)); VARIANTS[f'{SIG_NAMES.get(s, s)} | walk-forward k'] = {'family': 'signal charge', 'setting': f'{SIG_NAMES.get(s, s)} walk-forward k', 'yM': yM, 'y0': y0}
print(f'{len(_sf)} signal settings quoted; walk-forward picks added | [7] {time.perf_counter() - _s_t0:.0f}s')
record('variants_signals', settings=list(_sf), coverage={k: v.get('coverage') for k, v in _sf.items()}, seconds=time.perf_counter() - _s_t0)

## 8. The matrix

Every variant in pairs A, B and C, raw and levelled, with the bars of `variants_bars` and the four labels. The leaderboards are per pair and mode,
ranked by the Newey-West t under the exit of record; the sign under the guard and the common markout sits beside every row.

In [ ]:
CELL_T('8. The matrix: pairs A, B, C raw and levelled [8]')
_x_t0 = time.perf_counter(); _rows_vm = []
for nm, v_ in VARIANTS.items():
    for pair, yv, yb, reg, fr, bk in (('A', v_['yM'], Y_PM, 'M', FRONTIER['M'], BK_BASE['M']), ('B', v_['y0'], Y_PM, '0', FRONTIER['M'], BK_BASE['M']), ('C', v_['y0'], Y_P0, '0', FRONTIER['0'], BK_BASE['0'])):
        if not np.isfinite(yv).any():
            continue
        _rows_vm.append(score_pair(nm, yv, yb, fr, family=v_['family'], setting=v_['setting'], regime=reg, pair=pair, mode='raw', bk_base=bk))
        yl, wd = level_to(yv, yb, np.flatnonzero(_tmw & np.isfinite(yv) & np.isfinite(yb)))
        _rows_vm.append(score_pair(nm, yl, yb, None, family=v_['family'], setting=v_['setting'], regime=reg, pair=pair, mode='levelled', widths=wd, bk_base=bk))
VM = pd.DataFrame(_rows_vm)
_need = np.minimum(CFG.bar_months, VM['months'])
_raises = VM['win rate'] > VM['win rate, base'] + 1e-9
_c_rec = (VM['NW t | record'] >= CFG.bar_t) & (VM['months > 0 | record'] >= _need) & (VM['above width frontier (bp)'].fillna(-1) >= 0)
_c_guard = (VM['NW t | guard'] >= CFG.bar_t) & (VM['months > 0 | guard'] >= _need)
_c_common = ~_raises | ((VM['NW t | common markout'] >= CFG.bar_t) & (VM['months > 0 | common markout'] >= _need))
_c_lev = (VM['wins kept'] >= CFG.bar_wins_kept) & (VM['bp saved on kept cover wins'] >= CFG.bar_cover_saved_bp) & (VM['delta | hedged (bp)'] >= 0) & (VM['months > 0 | hedged'] >= _need)
VM['gate'] = np.where(VM['mode'] == 'raw', _c_rec & _c_guard & _c_common, _c_lev)
VM['conditions failed'] = [', '.join(c for c, ok in (('record bar', r), ('guard', g), ('common markout', cm)) if not ok) if md == 'raw' else ', '.join(c for c, ok in (('wins kept >= 80%', wk), ('cover saved >= 1 bp', cs), ('hedged delta >= 0', hd)) if not ok)
                           for md, r, g, cm, wk, cs, hd in zip(VM['mode'], _c_rec, _c_guard, _c_common, VM['wins kept'] >= CFG.bar_wins_kept, VM['bp saved on kept cover wins'] >= CFG.bar_cover_saved_bp, VM['delta | hedged (bp)'] >= 0)]
VM['conditions failed'] = VM['conditions failed'].replace('', '-')
_flip = ((np.sign(VM['delta | guard (bp)']) != np.sign(VM['delta | record (bp)'])) & (VM['NW t | guard'].abs() >= 2)) | ((np.sign(VM['delta | common markout (bp)']) != np.sign(VM['delta | record (bp)'])) & (VM['NW t | common markout'].abs() >= 2))
VM['label'] = [label4(g, d, t, f) for g, d, t, f in zip(VM['gate'], VM['delta | record (bp)'], VM['NW t | record'], _flip)]
_show = ['family', 'win rate', 'win rate, base', 'wins kept', 'new wins', 'wins given up', 'bp left on table (won, cover)', 'bp saved on kept cover wins', 'crossing on new wins (bp)', 'levelling width (bp)',
         'delta | record (bp)', 'NW t | record', 'months > 0 | record', 'delta | guard (bp)', 'NW t | guard', 'delta | common markout (bp)', 'NW t | common markout', 'delta | hedged (bp)', 'NW t | hedged',
         'delta | other side (bp)', 'delta $ per month ($k)', 'above width frontier (bp)', 'ex-ante sd vs base', 'gross DV01 ($k/bp)', 'conditions failed', 'label']
VERDICTS = {}
for pair, q in (('A', 'variant with the manual charge against production'), ('B', 'variant without the manual charge against production'), ('C', 'both without the manual charge')):
    for mode in ('raw', 'levelled'):
        t = VM[(VM['pair'] == pair) & (VM['mode'] == mode)].set_index('variant').sort_values('NW t | record', ascending=False, na_position='last')
        acc = t.index[t['gate']].tolist(); pos = t.index[t['label'].str.startswith('positive')].tolist()
        VERDICTS[f'{pair} {mode}'] = f'{len(acc)} of {len(t)} accepted' + (f': {acc}' if acc else '') + f'; positive evidence, gate not met: {pos if pos else "none"}; labels {t["label"].value_counts().to_dict()}'
        _top = t.index[:30].union(t.index[t['gate'] | t['label'].str.startswith('positive')], sort=False)
        print(f'\nPAIR {pair} {mode.upper()} ({q}; ranked by NW t under the exit of record; {len(_top)} of {len(t)} rows shown, the full matrix is in variant_matrix.parquet):'); display(t.loc[_top, _show].round(4))
        print(f'VERDICT (variants pair {pair} {mode}): {VERDICTS[f"{pair} {mode}"]}')
print('\nThe reference pair D sits in Section 4:', {m: f'delta record {r["delta | record (bp)"]:+.3f} bp, win rate {r["win rate"]:.3f} vs {r["win rate, base"]:.3f}, cover saved {r["bp saved on kept cover wins"]:+.2f} bp' for m, r in PAIR_D.iterrows()})
_lc = {'accepted under the specified gate': '#55A868', 'positive evidence, gate not met': '#4C72B0', 'inconclusive': '#BBBBBB', 'negative evidence': '#C44E52'}
fig, ax = plt.subplots(1, 3, figsize=(22, 6))
a = VM[(VM['pair'] == 'A') & (VM['mode'] == 'raw')]
ax[0].scatter(a['delta | record (bp)'], a['delta | guard (bp)'], c=[_lc.get(l_, '#999999') for l_ in a['label']], s=28); ax[0].axhline(0, color='k', lw=0.5); ax[0].axvline(0, color='k', lw=0.5)
ax[0].plot([a['delta | record (bp)'].min(), a['delta | record (bp)'].max()], [a['delta | record (bp)'].min(), a['delta | record (bp)'].max()], ls='--', color='#999999', lw=0.8)
ax[0].set_xlabel('delta vs production, exit of record (bp)'); ax[0].set_ylabel('delta vs production, guard (bp)'); ax[0].set_title('Pair A raw: every variant under the record exit and the guard (colour = label)', fontsize=9)
for pair, mk in (('B', 'o'), ('C', 's')):
    b = VM[(VM['pair'] == pair) & (VM['mode'] == 'levelled')]
    ax[1].scatter(b['wins kept'], b['bp saved on kept cover wins'], marker=mk, c=[_lc.get(l_, '#999999') for l_ in b['label']], s=28, label=f'pair {pair}')
ax[1].axvline(CFG.bar_wins_kept, color='#999999', ls='--', lw=0.8); ax[1].axhline(CFG.bar_cover_saved_bp, color='#999999', ls='--', lw=0.8); ax[1].set_xlabel('wins kept (share of the comparator\'s wins)'); ax[1].set_ylabel('cover saved on kept cover wins (bp)')
ax[1].set_title('Pairs B and C levelled: wins kept against cover saved (dashed = the bar)', fontsize=9); ax[1].legend(fontsize=8)
_fam = VM[(VM['pair'] == 'A') & (VM['mode'] == 'raw')].groupby('family')['delta | record (bp)'].agg(['max', 'median', 'min'])
ax[2].barh(_fam.index, _fam['max'], color='#55A868', label='best setting'); ax[2].barh(_fam.index, _fam['median'], color='#937860', label='median setting'); ax[2].axvline(0, color='k', lw=0.6)
ax[2].set_title('Pair A raw: delta under the exit of record by family (bp per request)', fontsize=9); ax[2].legend(fontsize=8); ax[2].tick_params(axis='y', labelsize=8)
savefig('v8_matrix')
record('variants_matrix', table=flat_records(VM.set_index(['variant', 'pair', 'mode'])), verdicts=VERDICTS, seconds=time.perf_counter() - _x_t0)
VM.to_parquet(OUT / 'variant_matrix.parquet')
print(f'matrix written: {OUT / "variant_matrix.parquet"} ({len(VM)} rows) | [8] {time.perf_counter() - _x_t0:.0f}s')

## 9. Registry and runtimes

In [ ]:
CELL_T('9. Registry and runtimes [9]')
CELL_T('end')
ct = pd.DataFrame(CELL_LOG); ct['share'] = ct['seconds'] / ct['seconds'].sum()
if CACHE_LOG:
    _cl = pd.DataFrame(CACHE_LOG); ct['cached stages'] = ct['cell'].map(_cl.groupby('cell').apply(lambda g: ', '.join(f'{r.stage} ({r.status}, {r.seconds:.0f}s)' for r in g.itertuples()), include_groups=False)).fillna('')
print(f'Runtime: {ct["seconds"].sum() / 60:.1f} min over {len(ct)} cells'); display(ct.set_index('cell').round(1))
record('runtime', total_seconds=float(ct['seconds'].sum()), cells=ct.round(2).to_dict(orient='records'))
(OUT / 'results_registry_variants.json').write_text(json.dumps(REGISTRY, indent=2, default=str), encoding='utf-8')
print('registry updated:', OUT / 'results_registry_variants.json')